## Setup

Import and merge files

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from mlxtend.frequent_patterns import apriori, association_rules

customer = pd.read_csv("customer.csv")
sale = pd.read_csv('sale.csv')
location = pd.read_csv('location.csv')
product = pd.read_csv("product.csv")

customer_sale = customer.merge(sale, on='customer_id',how='right')
customer_sale = pd.DataFrame(customer_sale)
customer_sale_prodduct = customer_sale.merge(product, on='product_id', how='left')
customer_sale_prodduct = pd.DataFrame(customer_sale_prodduct)
all_merged = customer_sale_prodduct.merge(location, on='city_id',how='left')
all_merged = pd.DataFrame(all_merged)

Clean merged file.  Added month and value column, removed 3 region columns

In [ ]:
all_merged['sale_date'] = pd.to_datetime(all_merged['sale_date'], format="%Y-%m-%d") 
all_merged['month'] = all_merged['sale_date'].dt.month
all_merged['product_category'] = all_merged['product_category'].str.strip().str.lower()

conditions = [
    all_merged['region'] == 'West',
    all_merged['region'] == 'Northeast',
    all_merged['region'] == 'South'
]

choices = [all_merged['West'], all_merged['Northeast'], all_merged['South']]
all_merged['value'] = np.select(conditions, choices, default=np.nan)
all_merged['value'] = all_merged['value'].str.replace('$', '', regex=False).str.replace(',', '', regex=False)
all_merged['value'] = all_merged['value'].astype(float)

all_merged.drop(columns=['West','Northeast','South'], inplace=True)
all_merged = all_merged.drop_duplicates()


Added a 'quarter' column

In [ ]:
Q1 = [1, 2, 3]
Q2 = [4, 5, 6]
Q3 = [7, 8, 9]
Q4 = [10, 11, 12]
category_map = {}

Q_combined = {}
for item in Q1:
    category_map[item] = 'Q1'
for item in Q2:
    category_map[item] = 'Q2'
for item in Q3:
    category_map[item] = 'Q3'
for item in Q4:
    category_map[item] = 'Q4'

all_merged['quarter'] = all_merged['month'].map(category_map)

### Counts, Groupings, and Initial findings

In [ ]:
# Quarter/monthly revenue and quarterly/monthly revenue by region

quarter_summary = all_merged.groupby('quarter')['value'].sum()
quarter_region_summary = all_merged.groupby(['region','quarter'])['value'].sum()
monthly_revenue = all_merged.groupby('month')['value'].sum()
region_monthly_revenue = all_merged.groupby(['region','month'])['value'].sum()


print(f'Revenue per quarter:')
print(quarter_summary.to_string())
print(f'Regional revenue per quarter:')
print(quarter_region_summary.to_string())
print(f'Total revenue per month:')
print(monthly_revenue.to_string())
print(f'Revenue by region per month:')
print(region_monthly_revenue.to_string())

In [ ]:
# Product revenue and units sold per quarter

product_region_revenue = all_merged.groupby(['product_category','quarter'])['value'].sum()
product_region_units = all_merged.groupby(['product_category','quarter'])['value'].count()

print(f'Product revenue per quarter:')
print(product_region_revenue.to_string())
print(f'Product units sold per quarter:')
print(product_region_units.to_string())

In [ ]:
# Number of members per region and their grouped average spend

customer_region_count = all_merged.groupby('region')['customer_id'].nunique()
customer_grouped_avg_purchase = all_merged.groupby(['region', 'customer_id'])['value'].sum().groupby('region').mean()

city_monthly_totals = all_merged.groupby(['region', 'city_name', 'month'])['value'].sum()
regional_monthly_avg = city_monthly_totals.groupby(['region', 'city_name']).mean().groupby('region').mean()


print(f'Number of customers per region:')
print(customer_region_count.to_string())
print(f'Average customer spend per region:')
print(customer_grouped_avg_purchase.to_string())
print(f'Average monthly revenue by region:')
print(regional_monthly_avg.to_string())


In [ ]:
# Regional product sales

region_category = all_merged.groupby(['region','product_category'])['value'].sum()

print(f'Revenue by region and product category:')
print(region_category.to_string())

In [ ]:
# City revenue, and city revenue by month

city_revenue = all_merged.groupby('city_name')['value'].sum()
city_monthly_money = all_merged.groupby(['city_name','month'])['value'].sum()
pd.set_option('display.max_rows', None)

print(f'Revenue per city:')
print(city_revenue.to_string())
print(f'Monthly revenue by city:')
print(city_monthly_money)

In [ ]:
# Average transactions, unit purchases

city_summary = all_merged.groupby('city_name').agg(
    avg_transactions=('sale_id', 'nunique'),
    avg_item_purchases=('product_id', 'sum'),
    avg_item_count_per_transaction=('product_id', 'mean')
).reset_index()

print(city_summary)

In [ ]:
# Units sold, transactions, and avg spend by city

city_units_sold = all_merged.groupby('city_name')['product_category'].value_counts()
city_transactions = all_merged.groupby('city_name')['sale_id'].nunique()
city_avg_purchase_revenue = all_merged.groupby('city_name')['value'].mean()


print(f'Units sold by category per city:')
print(city_units_sold.to_string())
print(f'Number of transactions per city:')
print(city_transactions.to_string())
print(f'Average customer spend per transaction by city:')
print(city_avg_purchase_revenue.to_string())

In [ ]:
# Total category revenue, units sold, average price

category_summary = all_merged.groupby('product_category').agg(
    revenue=('value', 'sum'),
    units_sold=('product_id', 'count'),
    avg_price=('value', 'mean')
).reset_index()

print(f'Category summary:')
print(category_summary)

In [ ]:
# Total category revenue, units sold, and average price by month

monthly_category_summary = all_merged.groupby(['product_category','month']).agg(
    revenue=('value', 'sum'),
    units_sold=('product_id', 'count'),
    avg_price=('value', 'mean')
).reset_index()

print(monthly_category_summary)

In [ ]:
# Total individual product revenue and units sold

product_revenue_units_sold = (
    all_merged.groupby('product_name').agg(
        revenue=('value', 'sum'), 
        units_sold=('product_id', 'count'))
    .sort_values('revenue', ascending=False)
)
print(f'Product revenue and units sold:')
print(product_revenue_units_sold)

### Basket Analysis

In [ ]:
basket_size = all_merged.groupby('sale_id')['sale_date'].count()
basket_value = all_merged.groupby('sale_id')['value'].sum()

print(basket_size.describe())
print(basket_value.describe())

In [ ]:
# Percent of Revenue by Product

product_summary = (all_merged.groupby("product_name").agg(revenue=("value", "sum")).reset_index())
total_revenue = product_summary["revenue"].sum()

product_summary["revenue_pct"] = (product_summary["revenue"] / total_revenue) * 100

product_summary.round(2).sort_values(by='revenue_pct', ascending=False)

In [ ]:
# Built out df for plotting frequency / revenue of products

product_plot = (all_merged.groupby('product_name').agg(frequency=('product_id', 'count'),revenue=('value', 'sum')).reset_index())

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(product_plot['frequency'] -20, product_plot['revenue'] + 50)

for _, row in product_plot.iterrows():
    plt.text(
        row['frequency'],
        row['revenue'],
        row['product_name'],
        fontsize=5)

plt.xlabel('Units Sold')
plt.ylabel('Revenue ($)')
plt.title('Frequency vs Revenue by Product')
plt.show()

In [ ]:
# First plot included most products in one area; broke out into a separate plot.

filtered_prod_list = ['Coffee Art print (4"x6")','Avocado Toast','Breakfast Sandwich','Nitro Cold Brew','Flat White','Matcha Green Tea Latte','Chai Tea Latte','Chocolate Croissant','Butter Croissant','Espresso','Banana Nut Bread','Cotado','Amricano','Drip Coffee','Chocolate Chip Cookie','Espresso','Muffin','Cortado','Chocolate Croissant','Cold Brew Coffee','Americano']

filtered_summary = (all_merged.groupby('product_name').agg(frequency=('product_id', 'count'),revenue=('value', 'sum')).reset_index())
filtered_summary = product_plot[product_plot['product_name'].isin(filtered_prod_list)]

plt.figure(figsize=(10,6))
plt.scatter(filtered_summary['frequency'] +5, filtered_summary['revenue'] + 120)

for _, row in filtered_summary.iterrows():
    plt.text(
        row['frequency'],
        row['revenue'],
        row['product_name'],
        fontsize=7)

plt.xlabel('Units Sold')
plt.ylabel('Revenue ($)')
plt.title('Frequency vs Revenue by Product')
plt.show()

In [ ]:
# Apriori algorithm to generate association rules between categories

basket_category = (all_merged.groupby(['sale_id', 'product_category']).size().unstack(fill_value=0))
basket_category = basket_category.astype(bool)

frequent_items = apriori(basket_category, min_support=0.001, use_colnames=True)

rules = association_rules(frequent_items, metric="lift", min_threshold=0.002)

strong_rules = rules[(rules['support'] >= 0.01) & (rules['confidence'] >= 0.05) & (rules['lift'] >= 1.3)].copy()
strong_rules['antecedents'] = (strong_rules['antecedents'].apply(lambda x: ', '.join(list(x))))
strong_rules['consequents'] = (strong_rules['consequents'].apply(lambda x: ', '.join(list(x))))

market_basket_results = (strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'conviction']]
    .sort_values(by='lift', ascending=False))

market_basket_results.round(3)

In [ ]:
def run_chi_square_and_odds_ratio(df, col_a, col_b):
    contingency_table = pd.crosstab(df[col_a], df[col_b])
    
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    a = contingency_table.loc[False, False]
    b = contingency_table.loc[False, True]
    c = contingency_table.loc[True, False]
    d = contingency_table.loc[True, True]
    
    odds_ratio = (d * a) / (b * c)
    
    n = contingency_table.values.sum()
    phi = np.sqrt(chi2 / n)
   
    if (a * d) < (b * c):
        phi = -phi
    
    significant = p_value < 0.05
    
    print(f"\n--- {col_a} vs {col_b} ---")
    print("Contingency Table:")
    print(contingency_table)
    print(f"Chi-square statistic: {chi2:.4f}")
    print(f"p-value: {p_value:.6f}")
    print(f"Degrees of freedom: {dof}")
    print(f"Significant at α=0.05? {'Yes' if significant else 'No'}")
    print(f"Odds Ratio: {odds_ratio:.4f}")
    print(f"Phi coefficient (r): {phi:.4f}")
    
    return {
        'pair': f"{col_a} vs {col_b}",
        'chi2': chi2,
        'p_value': p_value,
        'dof': dof,
        'significant': significant,
        'odds_ratio': odds_ratio,
        'phi': phi
    }

results = []
results.append(run_chi_square_and_odds_ratio(basket_category, 'drink', 'food'))
results.append(run_chi_square_and_odds_ratio(basket_category, 'drink', 'item'))
results.append(run_chi_square_and_odds_ratio(basket_category, 'food', 'item'))